# 01 — Konsep Dasar Object Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/03-object-detection/01_konsep_deteksi.ipynb)

Klasifikasi (modul 02) menjawab *"gambar ini berisi apa?"*. **Object detection** menjawab *"objek apa saja, dan di mana?"*. Output-nya berupa daftar **bounding box**, masing-masing dengan **label kelas** dan **skor keyakinan (confidence)**.

Notebook ini membangun konsep inti deteksi dari nol dengan NumPy, tanpa model:
1. Bounding box dan format-formatnya.
2. **IoU** (Intersection over Union), yang kita implementasikan sendiri.
3. **Non-Maximum Suppression (NMS)**.
4. **Precision, recall**, kurva PR, **AP**, dan **mAP**, dengan contoh angka.

Notebook ini tidak butuh GPU.

## 0. Setup

Hanya perlu NumPy dan Matplotlib. `torchvision` dipakai di beberapa tempat untuk memverifikasi bahwa implementasi kita benar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torchvision

np.set_printoptions(precision=3, suppress=True)
print("torchvision:", torchvision.__version__)

## 1. Bounding Box

Bounding box adalah persegi panjang yang membungkus objek. Ada beberapa format yang umum dipakai, dan salah format adalah sumber bug paling sering di proyek deteksi:

| Format | Isi | Dipakai oleh |
|---|---|---|
| `xyxy` | `x_min, y_min, x_max, y_max` (piksel) | Pascal VOC, torchvision, perhitungan IoU |
| `xywh` | `x_min, y_min, lebar, tinggi` (piksel) | COCO |
| `cxcywh` ternormalisasi | `x_tengah, y_tengah, lebar, tinggi` dibagi ukuran gambar (0–1) | YOLO |

Catatan: di gambar, sumbu **y mengarah ke bawah**, dan titik (0, 0) berada di pojok kiri atas.

Mari kita tulis fungsi konversi antar-format.

In [ ]:
def xyxy_to_xywh(b):
    x1, y1, x2, y2 = b
    return np.array([x1, y1, x2 - x1, y2 - y1])


def xyxy_to_yolo(b, img_w, img_h):
    # Pusat kotak, lalu semua nilai dibagi ukuran gambar agar berada di rentang 0–1
    x1, y1, x2, y2 = b
    return np.array([(x1 + x2) / 2 / img_w, (y1 + y2) / 2 / img_h,
                     (x2 - x1) / img_w, (y2 - y1) / img_h])


def yolo_to_xyxy(b, img_w, img_h):
    cx, cy, w, h = b
    return np.array([(cx - w / 2) * img_w, (cy - h / 2) * img_h,
                     (cx + w / 2) * img_w, (cy + h / 2) * img_h])


# Contoh: gambar 640×480, objek dari (100, 50) sampai (300, 250)
IMG_W, IMG_H = 640, 480
box = np.array([100, 50, 300, 250])
yolo = xyxy_to_yolo(box, IMG_W, IMG_H)

print("xyxy        :", box)
print("xywh (COCO) :", xyxy_to_xywh(box))
print("YOLO        :", yolo)
print("YOLO → xyxy :", yolo_to_xyxy(yolo, IMG_W, IMG_H))  # harus kembali ke nilai awal

Fungsi bantu untuk menggambar kotak. Fungsi ini akan dipakai berulang kali di notebook ini.

In [ ]:
def draw_boxes(ax, boxes, color="red", labels=None, lw=2, ls="-"):
    # boxes: array (N, 4) dalam format xyxy
    for i, (x1, y1, x2, y2) in enumerate(np.atleast_2d(boxes)):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor=color, lw=lw, ls=ls))
        if labels is not None:
            ax.text(x1, y1 - 3, labels[i], color=color, fontsize=9, va="bottom")


def empty_canvas(ax, w=IMG_W, h=IMG_H, title=""):
    ax.set_xlim(0, w); ax.set_ylim(h, 0)  # sumbu y dibalik: 0 di atas
    ax.set_aspect("equal"); ax.set_title(title)


fig, ax = plt.subplots(figsize=(5, 4))
empty_canvas(ax, title="Bounding box (100, 50) → (300, 250)")
draw_boxes(ax, box, labels=["objek"])
ax.plot(*((box[:2] + box[2:]) / 2), "r+", ms=12)  # titik pusat (dipakai format YOLO)
plt.show()

## 2. IoU (Intersection over Union)

IoU mengukur seberapa besar dua kotak saling tumpang tindih:

$$\text{IoU} = \frac{\text{luas irisan}}{\text{luas gabungan}} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}$$

- IoU = 1 → kedua kotak identik; IoU = 0 → tidak bersentuhan.
- Dipakai untuk: menentukan apakah prediksi **cocok** dengan ground truth (biasanya IoU ≥ 0.5), dan di dalam **NMS**.

Langkah menghitung irisan: ambil `max` dari koordinat kiri-atas dan `min` dari koordinat kanan-bawah. Jika lebar/tinggi irisan negatif, berarti tidak ada irisan (di-*clip* ke 0).

In [ ]:
def box_area(b):
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])


def iou(a, b):
    # Koordinat kotak irisan
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    irisan = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    gabungan = box_area(a) + box_area(b) - irisan
    return irisan / gabungan if gabungan > 0 else 0.0


# Contoh angka yang bisa dicek manual:
# A = (0,0)-(10,10) luas 100, B = (5,5)-(15,15) luas 100
# irisan = (5,5)-(10,10) = 25, gabungan = 100 + 100 - 25 = 175 → IoU = 25/175 ≈ 0.143
print("IoU contoh manual:", round(iou([0, 0, 10, 10], [5, 5, 15, 15]), 3))

Beberapa pasangan kotak dengan tingkat tumpang tindih berbeda. Perhatikan bahwa kotak yang terlihat "cukup mirip" pun bisa punya IoU di bawah 0.5. IoU adalah ukuran yang cukup ketat.

In [ ]:
pasangan = [
    ("identik",        [100, 100, 300, 300], [100, 100, 300, 300]),
    ("bergeser",       [100, 100, 300, 300], [150, 130, 350, 330]),
    ("bergeser jauh",  [100, 100, 300, 300], [220, 200, 420, 400]),
    ("di dalam",       [100, 100, 300, 300], [150, 150, 250, 250]),
    ("terpisah",       [100, 100, 200, 200], [300, 250, 400, 350]),
]

fig, axes = plt.subplots(1, len(pasangan), figsize=(16, 3.5))
for ax, (nama, a, b) in zip(axes, pasangan):
    nilai = iou(a, b)
    empty_canvas(ax, 500, 450, f"{nama}\nIoU = {nilai:.3f}")
    draw_boxes(ax, a, "tab:blue")
    draw_boxes(ax, b, "tab:red", ls="--")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

### 2.1 Versi vektor (banyak kotak sekaligus)

Dalam praktik kita menghitung IoU antara **N** kotak dan **M** kotak sekaligus, menghasilkan matriks `N × M`. Dengan *broadcasting* NumPy (`[:, None]` menambah dimensi), tidak perlu loop Python. Hasilnya kita cocokkan dengan `torchvision.ops.box_iou`.

In [ ]:
def iou_matrix(boxes_a, boxes_b):
    a = np.asarray(boxes_a, dtype=float)[:, None, :]  # (N, 1, 4)
    b = np.asarray(boxes_b, dtype=float)[None, :, :]  # (1, M, 4)
    x1 = np.maximum(a[..., 0], b[..., 0]); y1 = np.maximum(a[..., 1], b[..., 1])
    x2 = np.minimum(a[..., 2], b[..., 2]); y2 = np.minimum(a[..., 3], b[..., 3])
    irisan = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    luas_a = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    luas_b = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return irisan / (luas_a + luas_b - irisan)


A = np.array([p[1] for p in pasangan])
B = np.array([p[2] for p in pasangan])
kita = iou_matrix(A, B)
tv = torchvision.ops.box_iou(torch.tensor(A, dtype=torch.float32), torch.tensor(B, dtype=torch.float32)).numpy()

print("Matriks IoU (baris = kotak biru, kolom = kotak merah):\n", kita)
print("Diagonal (= 5 pasangan di gambar atas):", np.diag(kita))
print("Sama dengan torchvision?", np.allclose(kita, tv, atol=1e-6))

## 3. Non-Maximum Suppression (NMS)

Detektor biasanya menghasilkan **banyak kotak yang saling tumpang tindih** untuk objek yang sama. NMS menyaring kotak-kotak itu hingga tersisa satu kotak terbaik per objek:

1. Buang kotak dengan confidence di bawah ambang (`conf_thr`).
2. Urutkan kotak berdasarkan confidence, dari tertinggi.
3. Ambil kotak teratas → **simpan**.
4. Hapus semua kotak tersisa yang IoU-nya dengan kotak tersebut ≥ `iou_thr` (dianggap duplikat).
5. Ulangi langkah 3–4 sampai tidak ada kotak tersisa.

Pada praktiknya NMS dijalankan **per kelas**, agar kotak "orang" tidak menghapus kotak "sepeda" yang tumpang tindih dengannya.

In [ ]:
def nms(boxes, scores, iou_thr=0.5, conf_thr=0.0):
    boxes, scores = np.asarray(boxes, float), np.asarray(scores, float)
    idx = np.where(scores >= conf_thr)[0]
    idx = idx[np.argsort(-scores[idx])]  # urutkan confidence tertinggi dulu
    keep = []
    while len(idx) > 0:
        terbaik = idx[0]
        keep.append(int(terbaik))
        sisa = idx[1:]
        ious = iou_matrix(boxes[[terbaik]], boxes[sisa])[0]
        idx = sisa[ious < iou_thr]  # buang duplikat
    return keep


# Contoh: dua objek, masing-masing punya beberapa prediksi yang saling tumpang tindih
pred_boxes = np.array([
    [ 50,  60, 250, 300],   # 0: objek A
    [ 60,  50, 265, 290],   # 1: objek A (duplikat)
    [ 40,  75, 240, 310],   # 2: objek A (duplikat)
    [330, 120, 560, 380],   # 3: objek B
    [345, 110, 580, 395],   # 4: objek B (duplikat)
    [200, 330, 300, 430],   # 5: kotak lain, confidence rendah
])
pred_scores = np.array([0.92, 0.85, 0.71, 0.88, 0.64, 0.30])

print("IoU antar prediksi:\n", iou_matrix(pred_boxes, pred_boxes))
keep = nms(pred_boxes, pred_scores, iou_thr=0.5, conf_thr=0.4)
print("\nIndeks yang disimpan NMS:", keep)
print("torchvision.ops.nms     :",
      torchvision.ops.nms(torch.tensor(pred_boxes[pred_scores >= 0.4], dtype=torch.float32),
                          torch.tensor(pred_scores[pred_scores >= 0.4], dtype=torch.float32), 0.5).tolist())

Visualisasi sebelum dan sesudah NMS. Kotak nomor 5 hilang karena confidence-nya di bawah `conf_thr = 0.4`, bukan karena tumpang tindih.

In [ ]:
label_semua = [f"#{i} {s:.2f}" for i, s in enumerate(pred_scores)]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
empty_canvas(ax[0], title=f"Sebelum NMS ({len(pred_boxes)} kotak)")
draw_boxes(ax[0], pred_boxes, "tab:red", label_semua, lw=1.5)
empty_canvas(ax[1], title=f"Sesudah NMS ({len(keep)} kotak)")
draw_boxes(ax[1], pred_boxes[keep], "tab:green", [label_semua[i] for i in keep])
plt.tight_layout()
plt.show()

**Efek `iou_thr`:**
- Terlalu **rendah** (misal 0.2) → objek berbeda yang berdekatan bisa ikut terhapus (misalnya orang-orang di kerumunan).
- Terlalu **tinggi** (misal 0.9) → duplikat lolos, satu objek terdeteksi berkali-kali.

In [ ]:
for thr in [0.1, 0.3, 0.5, 0.7, 0.9]:
    k = nms(pred_boxes, pred_scores, iou_thr=thr, conf_thr=0.4)
    print(f"iou_thr = {thr:.1f} → {len(k)} kotak tersisa: {k}")

## 4. Precision dan Recall untuk Deteksi

Setelah NMS, setiap prediksi dicocokkan dengan ground truth (GT):
- **True Positive (TP)**: prediksi dengan IoU ≥ ambang (misal 0.5) terhadap GT yang **belum** dipasangkan, dan kelasnya benar.
- **False Positive (FP)**: prediksi yang tidak cocok dengan GT mana pun, atau duplikat dari GT yang sudah dipasangkan.
- **False Negative (FN)**: GT yang tidak terdeteksi.

$$\text{Precision} = \frac{TP}{TP + FP} \quad (\text{dari yang diprediksi, berapa yang benar?})$$
$$\text{Recall} = \frac{TP}{TP + FN} \quad (\text{dari objek yang ada, berapa yang ditemukan?})$$

Pencocokan dilakukan secara **greedy**: prediksi diproses dari confidence tertinggi, dan setiap GT hanya boleh dipasangkan sekali.

### 4.1 Contoh data

Misalkan ada 2 gambar dengan total **5 objek** (kelas "mobil"). Model menghasilkan 8 prediksi.

In [ ]:
# Ground truth per gambar (xyxy)
gt = {
    "img1": np.array([[ 30,  40, 200, 180], [260,  60, 420, 200], [450, 250, 600, 400]]),
    "img2": np.array([[ 80, 100, 300, 280], [350, 150, 520, 330]]),
}
# Prediksi: (gambar, kotak, confidence)
preds = [
    ("img1", [ 35,  45, 195, 190], 0.95),  # cocok GT img1-0
    ("img2", [ 90, 110, 310, 270], 0.91),  # cocok GT img2-0
    ("img1", [265,  70, 400, 210], 0.83),  # cocok GT img1-1
    ("img1", [ 40,  35, 210, 175], 0.77),  # duplikat GT img1-0 → FP
    ("img2", [500, 330, 620, 450], 0.64),  # tidak ada objek → FP
    ("img1", [470, 280, 640, 440], 0.52),  # IoU dengan GT img1-2 terlalu kecil? cek!
    ("img2", [355, 160, 515, 320], 0.45),  # cocok GT img2-1
    ("img1", [150, 300, 260, 420], 0.20),  # tidak ada objek → FP
]
N_GT = sum(len(v) for v in gt.values())
print("Total ground truth:", N_GT, "| total prediksi:", len(preds))

### 4.2 Mencocokkan prediksi dengan ground truth

In [ ]:
def match_predictions(preds, gt, iou_thr=0.5):
    # Kembalikan array TP (1) / FP (0) untuk prediksi yang sudah diurutkan berdasarkan confidence
    urut = sorted(preds, key=lambda p: -p[2])
    sudah_dipakai = {k: np.zeros(len(v), dtype=bool) for k, v in gt.items()}
    hasil = []
    for img, box, conf in urut:
        ious = iou_matrix([box], gt[img])[0] if len(gt[img]) else np.array([])
        j = int(np.argmax(ious)) if len(ious) else -1
        if j >= 0 and ious[j] >= iou_thr and not sudah_dipakai[img][j]:
            sudah_dipakai[img][j] = True
            hasil.append((img, conf, ious[j], 1))
        else:
            hasil.append((img, conf, ious[j] if j >= 0 else 0.0, 0))
    return hasil


cocok = match_predictions(preds, gt, iou_thr=0.5)
tp = np.array([h[3] for h in cocok])
cum_tp, cum_fp = np.cumsum(tp), np.cumsum(1 - tp)
precision = cum_tp / (cum_tp + cum_fp)
recall = cum_tp / N_GT

print(f"{'rank':>4} {'img':>5} {'conf':>5} {'IoU':>5} {'TP/FP':>6} {'cumTP':>6} {'cumFP':>6} {'prec':>6} {'recall':>7}")
for i, (img, conf, io, t) in enumerate(cocok):
    print(f"{i+1:>4} {img:>5} {conf:>5.2f} {io:>5.2f} {'TP' if t else 'FP':>6} "
          f"{cum_tp[i]:>6} {cum_fp[i]:>6} {precision[i]:>6.2f} {recall[i]:>7.2f}")

Cara membaca tabel: setiap baris menjawab *"jika ambang confidence diletakkan tepat di prediksi ini, berapa precision dan recall-nya?"*
- Ambang **tinggi** (baris atas) → sedikit prediksi, precision tinggi, recall rendah.
- Ambang **rendah** (baris bawah) → banyak prediksi, recall naik, precision turun.

Perhatikan prediksi dengan conf 0.52: kotaknya bergeser cukup jauh sehingga IoU-nya di bawah 0.5 dan dihitung FP, padahal objeknya "hampir" terdeteksi. Hasilnya, GT tersebut menjadi FN.

## 5. Kurva Precision–Recall dan Average Precision (AP)

Titik-titik (recall, precision) di atas membentuk **kurva PR**. **AP** adalah luas di bawah kurva tersebut: satu angka yang merangkum performa di **semua** ambang confidence.

Kurva mentah berbentuk zigzag, jadi dihaluskan dulu dengan **interpolasi**: precision di setiap recall diganti dengan precision **maksimum** di recall yang sama atau lebih tinggi. Hasilnya kurva yang selalu menurun (*envelope*), lalu luasnya dihitung. Ini metode *all-point interpolation* (Pascal VOC 2010+). COCO memakai varian 101 titik yang hasilnya sangat mirip.

In [ ]:
def average_precision(recall, precision):
    # Tambahkan titik awal (recall 0) dan akhir (recall 1)
    mrec = np.concatenate([[0.0], recall, [1.0]])
    mpre = np.concatenate([[1.0], precision, [0.0]])
    # Envelope: precision di titik i = max precision di recall >= recall_i
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    # Jumlahkan luas persegi panjang di titik-titik tempat recall berubah
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])), mrec, mpre


ap50, mrec, mpre = average_precision(recall, precision)
print(f"AP@0.5 = {ap50:.3f}")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(recall, precision, "o-", label="PR mentah")
ax.step(mrec, mpre, where="post", color="tab:red", label="interpolasi (envelope)")
ax.fill_between(mrec, mpre, step="post", alpha=0.15, color="tab:red", label=f"AP = {ap50:.3f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.05)
ax.set_title("Kurva Precision–Recall (IoU ≥ 0.5)"); ax.legend(loc="lower left")
plt.show()

## 6. mAP: mean Average Precision

"Mean" di sini punya dua arti yang sering digabung:

1. **Rata-rata antar kelas**: AP dihitung per kelas, lalu dirata-ratakan. Model yang bagus di kelas "orang" tapi buruk di "sepeda" akan terlihat dari sini.
2. **Rata-rata antar ambang IoU** (standar COCO): **mAP@0.5:0.95** adalah rata-rata AP pada ambang IoU 0.50, 0.55, …, 0.95 (10 ambang). Metrik ini lebih ketat karena menuntut kotak yang sangat presisi.

Istilah di laporan YOLO:
- `mAP50` = mAP pada IoU 0.5 (lebih "longgar").
- `mAP50-95` = rata-rata dari 0.5 sampai 0.95 (metrik utama COCO).

### 6.1 AP pada berbagai ambang IoU

In [ ]:
def ap_at(preds, gt, iou_thr):
    cocok = match_predictions(preds, gt, iou_thr)
    tp = np.array([h[3] for h in cocok])
    n_gt = sum(len(v) for v in gt.values())
    ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
    return average_precision(ctp / n_gt, ctp / (ctp + cfp))[0]


thresholds = np.arange(0.5, 0.96, 0.05)
aps = [ap_at(preds, gt, t) for t in thresholds]
for t, a in zip(thresholds, aps):
    print(f"AP@{t:.2f} = {a:.3f}  {'█' * int(a * 40)}")
print(f"\nAP@0.5:0.95 (kelas mobil) = {np.mean(aps):.3f}")

Semakin ketat ambang IoU, semakin banyak prediksi yang "kurang pas" berubah dari TP menjadi FP, sehingga AP turun.

### 6.2 mAP untuk beberapa kelas

Tambahkan kelas kedua ("orang") dengan data sendiri, lalu hitung mAP sebagai rata-rata AP kedua kelas.

In [ ]:
gt_orang = {
    "img1": np.array([[210, 250, 280, 440]]),
    "img2": np.array([[ 20, 300,  90, 470], [540,  40, 610, 220]]),
}
preds_orang = [
    ("img2", [ 25, 295,  95, 465], 0.90),  # TP
    ("img1", [300, 240, 380, 430], 0.75),  # FP (tidak ada orang di sana)
    ("img1", [205, 255, 285, 450], 0.60),  # TP
    ("img2", [530,  60, 600, 230], 0.40),  # TP (IoU cukup untuk 0.5)
]

per_kelas = {
    "mobil": (preds, gt),
    "orang": (preds_orang, gt_orang),
}
ringkasan = {}
for nama, (p, g) in per_kelas.items():
    ap_50 = ap_at(p, g, 0.5)
    ap_5095 = np.mean([ap_at(p, g, t) for t in thresholds])
    ringkasan[nama] = (ap_50, ap_5095)
    print(f"{nama:<6} AP50 = {ap_50:.3f} | AP50-95 = {ap_5095:.3f}")

map50 = np.mean([v[0] for v in ringkasan.values()])
map5095 = np.mean([v[1] for v in ringkasan.values()])
print(f"\nmAP50    = {map50:.3f}")
print(f"mAP50-95 = {map5095:.3f}")

## Ringkasan

- Kenali format kotak: `xyxy` (VOC/torchvision), `xywh` (COCO), `cxcywh` ternormalisasi (YOLO).
- **IoU** = irisan / gabungan; dipakai untuk mencocokkan prediksi dengan GT dan dalam NMS.
- **NMS** menghapus kotak duplikat: simpan kotak dengan confidence tertinggi, buang yang IoU-nya ≥ ambang.
- Prediksi diurutkan berdasarkan confidence, lalu dicocokkan secara greedy → TP/FP → kurva PR.
- **AP** = luas di bawah kurva PR (setelah interpolasi); **mAP** = rata-rata antar kelas (dan antar ambang IoU untuk mAP50-95).

## Latihan

1. **GIoU.** IoU bernilai 0 untuk semua pasangan kotak yang tidak bersentuhan, sehingga tidak bisa membedakan kotak yang "hampir kena" dan yang "jauh sekali". Implementasikan **Generalized IoU**: `GIoU = IoU − (luas C − luas gabungan) / luas C`, dengan C kotak terkecil yang membungkus A dan B. Uji pada pasangan "terpisah" di bagian 2 dan pada pasangan yang lebih jauh lagi. Verifikasi dengan `torchvision.ops.generalized_box_iou`.
2. **NMS per kelas dan Soft-NMS.** (a) Modifikasi `nms` agar menerima array `classes` dan hanya menekan kotak dari **kelas yang sama**. Buat contoh di mana kotak "orang" dan "sepeda" tumpang tindih dengan IoU 0.6 dan keduanya harus dipertahankan. (b) Implementasikan **Soft-NMS**: alih-alih menghapus, kalikan skor kotak yang tumpang tindih dengan `exp(-IoU² / σ)`. Bandingkan hasilnya di data contoh.
3. **Hitung manual lalu verifikasi.** Untuk kelas "orang" di bagian 6.2, tulis di kertas tabel TP/FP, precision, dan recall kumulatif pada IoU 0.5, lalu hitung AP secara manual dengan metode envelope. Setelah itu, ubah satu prediksi sehingga IoU-nya turun di bawah 0.5 dan prediksi dulu bagaimana AP50 akan berubah sebelum menjalankan kode.